# Federated Queries: Neo4j Aircraft Graph + Databricks Sensor Data

Combines graph topology from Neo4j with time-series sensor data from Databricks Delta tables.

**Data split:**
- **Neo4j** (graph-native): Aircraft, Systems, Sensors, Flights, Maintenance Events, Delays — all topology and operational data
- **Delta** (tabular): `sensor_readings` — 172,800 hourly readings from 160 sensors over 45 days

**Prerequisites:**
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j
- Run `01-simple-connect-test.ipynb` to create the UC JDBC connection and `sensor_readings` table

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these values for your environment
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://<instance>.databases.neo4j.io"
SECRET_SCOPE = "sample_validation"
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "aircraft_data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL_SQL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

In [ ]:
def read_neo4j(custom_schema, query):
    """Read from Neo4j through the UC JDBC connection."""
    return (
        spark.read.format("jdbc")
        .option("databricks.connection", UC_CONNECTION_NAME)
        .option("customSchema", custom_schema)
        .option("query", query)
        .load()
    )

---

## Query 1: Sensor Health by Aircraft

Joins Neo4j graph topology (aircraft → system → sensor) with Delta sensor statistics.

- **Neo4j JDBC**: Aircraft → HAS_SYSTEM → System → HAS_SENSOR → Sensor via NATURAL JOIN
- **Delta**: aggregate reading statistics per sensor_id
- **Join**: on `sensorId` in Spark

In [ ]:
print("=" * 60)
print("FEDERATED QUERY 1: Sensor Health by Aircraft")
print("=" * 60)

sensor_topology = read_neo4j(
    "aircraftId STRING, model STRING, systemType STRING, sensorId STRING",
    """SELECT a.aircraftId AS aircraftId, a.model AS model,
              sys.type AS systemType, s.sensorId AS sensorId
       FROM Aircraft a
       NATURAL JOIN HAS_SYSTEM r1
       NATURAL JOIN System sys
       NATURAL JOIN HAS_SENSOR r2
       NATURAL JOIN Sensor s""",
)

sensor_stats = spark.sql(f"""
    SELECT sensor_id,
           COUNT(*) AS reading_count,
           ROUND(AVG(value), 2) AS avg_value,
           ROUND(MIN(value), 2) AS min_value,
           ROUND(MAX(value), 2) AS max_value
    FROM {FQN}.sensor_readings
    GROUP BY sensor_id
""")

result = (
    sensor_topology
    .join(sensor_stats, sensor_topology.sensorId == sensor_stats.sensor_id, "left")
    .select("aircraftId", "model", "systemType", "sensorId", "reading_count", "avg_value")
)
result.orderBy("aircraftId", "systemType").show(10, truncate=False)
row_count = result.count()
status = "PASS" if row_count == 160 else "FAIL"
print(f"  [{status}] {row_count} sensor+aircraft rows")

---

## Query 2: Maintenance Severity and Sensor Health

Combines Neo4j maintenance event counts with Delta sensor averages per aircraft.

- **Neo4j JDBC**: maintenance event counts by severity; counts per aircraft
- **Delta**: average sensor value per aircraft derived from `sensor_id` prefix
- **Join**: on `aircraftId` in Spark

In [ ]:
print("=" * 60)
print("FEDERATED QUERY 2: Maintenance Severity + Sensor Health")
print("=" * 60)

# Neo4j: maintenance counts by severity
df_severity = read_neo4j(
    "severity STRING, event_count LONG",
    "SELECT severity, COUNT(*) AS event_count FROM MaintenanceEvent GROUP BY severity",
)
print("\n  Maintenance events by severity:")
df_severity.orderBy("event_count", ascending=False).show(truncate=False)

# Neo4j: maintenance counts per aircraft
df_by_aircraft = read_neo4j(
    "aircraftId STRING, maint_count LONG",
    "SELECT aircraftId, COUNT(*) AS maint_count FROM MaintenanceEvent GROUP BY aircraftId",
)

# Delta: average sensor value per aircraft (sensor_id starts with aircraft ID)
aircraft_sensor_health = spark.sql(f"""
    SELECT REGEXP_EXTRACT(sensor_id, '^(AC[0-9]+)', 1) AS aircraftId,
           ROUND(AVG(value), 2) AS avg_sensor_reading
    FROM {FQN}.sensor_readings
    GROUP BY REGEXP_EXTRACT(sensor_id, '^(AC[0-9]+)', 1)
""")

result = df_by_aircraft.join(aircraft_sensor_health, "aircraftId", "left")
print("  Maintenance count + avg sensor reading per aircraft:")
result.orderBy("maint_count", ascending=False).show(10, truncate=False)
status = "PASS" if result.count() == 20 else "FAIL"
print(f"  [{status}] {result.count()} aircraft")

---

## Query 3: Flight Delay Analysis by Operator

Pure Neo4j graph analytics: flight counts per operator and delay statistics by cause.

- **Neo4j JDBC**: flight counts grouped by operator
- **Neo4j JDBC**: delay counts and average minutes grouped by cause

In [ ]:
print("=" * 60)
print("FEDERATED QUERY 3: Flight Delay Analysis by Operator")
print("=" * 60)

df_flights = read_neo4j(
    "operator STRING, flight_count LONG",
    "SELECT operator, COUNT(*) AS flight_count FROM Flight GROUP BY operator",
)
print("\n  Flights by operator:")
df_flights.orderBy("flight_count", ascending=False).show(truncate=False)

df_delays = read_neo4j(
    "cause STRING, delay_count LONG, avg_minutes DOUBLE",
    "SELECT cause, COUNT(*) AS delay_count, AVG(minutes) AS avg_minutes FROM Delay GROUP BY cause",
)
print("  Delays by cause:")
df_delays.orderBy("delay_count", ascending=False).show(truncate=False)

print("Status: PASS — run 03-materialized-tables.ipynb next")